# 02 - Multiple Linear Regression End-to-End

This notebook teaches multiple linear regression using numeric and categorical predictors. It covers EDA, preprocessing, one-hot encoding, scikit-learn pipelines, statsmodels OLS, coefficient interpretation, visual diagnostics, and business-style conclusions.

## Learning objectives

Students will learn how to:

1. Define a multiple regression problem.
2. Separate numeric and categorical predictors.
3. Encode categorical variables using one-hot encoding.
4. Build a scikit-learn preprocessing + modelling pipeline.
5. Evaluate predictions using MAE, RMSE, and R².
6. Interpret coefficients using the phrase *holding other variables constant*.
7. Use statsmodels for inference tables.

## 1. Problem statement

We want to predict `sales_k_units` from several marketing and business drivers.

```text
sales_k_units = beta_0 + beta_1*tv_spend + beta_2*radio_spend + ... + error
```

Multiple regression asks: **What is the expected change in sales when one feature changes, holding other variables constant?**

## 2. Import libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.formula.api as smf

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

## 3. Load dataset

In [ ]:
DATA_DIR = Path.cwd().parent / 'data' if (Path.cwd().parent / 'data').exists() else Path.cwd() / 'data'
df = pd.read_csv(DATA_DIR / 'multiple_regression_marketing_sales.csv')
df.head()

## 4. Data audit

Before modelling, inspect data types, missing values, and unique values. This is especially important when a categorical variable like `season` is present.

In [ ]:
pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
    'missing_count': df.isna().sum().values,
    'n_unique': df.nunique().values
})

In [ ]:
df.describe(include='all').T

## 5. Visualize target by season

This checks whether sales differ meaningfully by season before the model learns seasonal effects.

In [ ]:
season_sales = df.groupby('season')['sales_k_units'].mean().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(season_sales.index, season_sales.values)
plt.title('Average sales by season')
plt.xlabel('Season')
plt.ylabel('Average sales, thousand units')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 6. Visualize numeric relationships

Each subplot compares one numeric predictor with the target. This helps students see which relationships are strong, weak, positive, or negative.

In [ ]:
numeric_features = ['tv_spend_k', 'radio_spend_k', 'social_spend_k', 'price_index', 'competitor_spend_k']
target = 'sales_k_units'

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for idx, feature in enumerate(numeric_features):
    axes[idx].scatter(df[feature], df[target], alpha=0.7)
    axes[idx].set_title(f'{feature} vs sales')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel(target)
    axes[idx].grid(True, alpha=0.3)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

## 7. Correlation heatmap

Correlation is not causation, but it is useful for quick relationship scanning and multicollinearity hints.

In [ ]:
corr = df[numeric_features + [target]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
ax.set_title('Correlation heatmap')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 8. Define features and target

We keep numeric and categorical columns separate so the preprocessing pipeline can treat them differently.

In [ ]:
categorical_features = ['season']
X = df[numeric_features + categorical_features]
y = df[target]

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)
print('Target:', target)

## 9. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=43
)

print('Training rows:', len(X_train))
print('Testing rows :', len(X_test))

## 10. Build preprocessing + regression pipeline

The categorical feature `season` is converted into dummy variables using one-hot encoding. The first category is dropped to avoid perfect dummy-variable multicollinearity.

In [ ]:
preprocess = ColumnTransformer([
    ('categorical', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
    ('numeric', 'passthrough', numeric_features),
])

pipeline = Pipeline([
    ('preprocess', preprocess),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

## 11. Evaluate model performance

In [ ]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

pd.DataFrame({
    'metric': ['MAE', 'RMSE', 'R2'],
    'value': [mae, rmse, r2]
})

## 12. Actual vs predicted plot

A strong model should place most points close to the diagonal line.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.8)
line_min = min(y_test.min(), pred.min())
line_max = max(y_test.max(), pred.max())
plt.plot([line_min, line_max], [line_min, line_max], linewidth=2)
plt.title('Actual vs predicted sales')
plt.xlabel('Actual sales')
plt.ylabel('Predicted sales')
plt.grid(True, alpha=0.3)
plt.show()

## 13. Coefficient extraction from pipeline

Coefficients explain model direction and magnitude. For multiple regression, interpret each coefficient **holding the other variables constant**.

In [ ]:
feature_names = pipeline.named_steps['preprocess'].get_feature_names_out()
coefficients = pipeline.named_steps['model'].coef_

coef_table = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', ascending=False)

coef_table

In [ ]:
coef_plot = coef_table.sort_values('coefficient')

plt.figure(figsize=(8, 5))
plt.barh(coef_plot['feature'], coef_plot['coefficient'])
plt.axvline(0, linewidth=1)
plt.title('Model coefficients')
plt.xlabel('Coefficient value')
plt.ylabel('Feature')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Prediction error table

This helps students inspect individual errors instead of only looking at aggregate metrics.

In [ ]:
error_table = X_test.copy()
error_table['actual_sales'] = y_test.values
error_table['predicted_sales'] = pred
error_table['residual'] = error_table['actual_sales'] - error_table['predicted_sales']
error_table['absolute_error'] = error_table['residual'].abs()
error_table.sort_values('absolute_error', ascending=False).round(2).head(10)

## 15. statsmodels OLS for inference

`statsmodels` gives p-values, confidence intervals, AIC, BIC, and a full OLS summary. This is more useful for statistical interpretation than pure prediction.

In [ ]:
ols = smf.ols(
    'sales_k_units ~ tv_spend_k + radio_spend_k + social_spend_k + price_index + competitor_spend_k + C(season)',
    data=df
).fit()

pd.DataFrame({
    'coefficient': ols.params,
    'p_value': ols.pvalues,
    'lower_95': ols.conf_int()[0],
    'upper_95': ols.conf_int()[1]
}).sort_values('p_value')

## 16. Cross-validation

Cross-validation evaluates the model across multiple data splits and gives a more stable view of model performance.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=43)
scores = cross_validate(
    pipeline, X, y, cv=cv,
    scoring={'r2': 'r2', 'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'}
)

pd.DataFrame({
    'fold': range(1, 6),
    'r2': scores['test_r2'],
    'mae': -scores['test_mae'],
    'rmse': -scores['test_rmse']
}).round(4)

## 17. Final conclusion template

A good conclusion should include:

1. Test metrics.
2. Strongest positive and negative drivers.
3. Whether results match business intuition.
4. Important limitations.
5. Whether the model is ready for prediction, explanation, or only learning/demo use.

## Student exercises

1. Remove `competitor_spend_k` and retrain. What happens to metrics?
2. Remove `season` and compare R².
3. Change the test size from 25% to 30%.
4. Add a residual histogram.
5. Write a business interpretation for `price_index` and `radio_spend_k`.